In [42]:
import pandas as pd
import joblib
import numpy as np
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import onnxruntime as ort
import numpy as np

In [43]:
# load unnormalised dataset
FEATURE_COLUMNS = ['d_front', 'd_back', 'v_front', 'v_back']
dataset = pd.read_csv('simulation-dataset.csv')
training_datas_unnormalised = dataset[FEATURE_COLUMNS].values

unnormalised_mins = training_datas_unnormalised.min(axis=0)
unnormalised_maxs = training_datas_unnormalised.max(axis=0)

details = pd.DataFrame({
    'Features': FEATURE_COLUMNS,
    'Data Min': unnormalised_mins,
    'Data Max': unnormalised_maxs
})
display(details)

# load scaler file and normalise
scaler = joblib.load('scaler.pkl')
training_datas = scaler.transform(training_datas_unnormalised)
print(f"\nFirst 5 lines of dataset:")

column_names = ['d_front (0)', 'd_back (1)', 'v_front (2)', 'v_back (3)']
training_data_df = pd.DataFrame(data=training_datas, columns=column_names)
display(training_data_df[:5])

LABEL_COLUMN = 'ego_a'
training_labels= dataset[LABEL_COLUMN].values
print(f"\nFirst 5 lines of labels:")
print(f"{training_labels[:5]}\n")

,Features,Data Min,Data Max
0,d_front,-0.092464,311.600923
1,d_back,-0.077204,305.828968
2,v_front,-5.563068,7.548104
3,v_back,-5.361264,7.422031



First 5 lines of dataset:


,d_front (0),d_back (1),v_front (2),v_back (3)
0,-0.444195,-0.894804,-0.183601,-0.362558
1,-0.444536,-0.894360,-0.349543,-0.128195
2,-0.445220,-0.892951,-0.350522,0.146128
3,-0.445917,-0.890531,-0.356197,0.405182
4,-0.446640,-0.887611,-0.362737,0.409462



First 5 lines of labels:
[0. 3. 3. 3. 3.]



In [44]:
# find normalisation rule
print(f"Scaler type: {scaler}")
details = pd.DataFrame({
        'features': FEATURE_COLUMNS,
        'Mean': scaler.mean_,
        'SD': scaler.scale_
    })
display(details)

Scaler type: StandardScaler()


,features,Mean,SD
0,d_front,68.479297,52.128595
1,d_back,69.437695,52.274740
2,v_front,0.393321,2.142253
3,v_back,0.718784,1.982537


Normalisation law:
$$X=(X_{unnorm} - mean) / SD$$

---
## Custom safety properties

### 1. Safe front
**Does the ego car have enough space to brake if the front car were to brake maximally?**

$$ pos_{front} - pos_{ego} > L $$

$$ (x_{front} + v_{front}^2 / 2B_{max}) - pos_{ego} > L $$
where $v_{front}^2 / 2B_{max}$ is the distance the front car would travel before stopping while braking maximally (note that this assumes $B_{max}$ is a negative value, which isn't the case in our code).

In metrics relative to the ego car, as per our data ($d_{front}$ is the distance from ego to front car, $v_{front}$ is the velocity difference between the two cars):

$$ d_{front} - v_{front}^2/2B_{max} > L $$

$$ d_{front} > L + v_{front}^2/2B_{max} $$


### 2. Safe back
**Does the ego car have enough space to accelerate sufficiently if the back car were to accelerate maximally?**

A symmetric analog to "safe front" (to keep things time-independent and conservative):

$$ pos_{ego} - pos_{back} > L $$

$$ pos_{ego} - (x_{back} - v_{back}^2 / 2A_{max}) > L $$

In metrics relative to the ego car::

$$ d_{back} - v_{back}^2/2A_{max} > L $$

$$ d_{back} > L + v_{back}^2/2A_{max} $$

---
## Marabou-friendly simplification ಥ_ಥ

Avoiding non-linearity by using constant values for $v_{front}$ and $v_{back}$, assuming worst-case:

$$ d_{front} > L + v_{max}^2/2B_{max} $$

$$ d_{back} > L + v_{min}^2/2A_{max} $$

---
## Final vehicle code!
```python
Bmax = 5.0
Amax = 3.0
Vmin = 0.0001
Vmax = 20.0
L = 4.0

safeFront : UnnormalisedInput -> Bool
safeFront x =
  x ! distanceToFrontCar > L and
  x ! distanceToFrontCar > L + (Vmax * Vmax)/(2 * Bmax)

safeBack : UnnormalisedInput -> Bool
safeBack x = 
  x ! distanceToBackCar > L and
  x ! distanceToBackCar > L + (Vmin * Vmin)/(2 * Amax)

@property
property1 : Bool
property1 = forall x .
  validInput x and
  not (safeFront x) =>
  actionToTake brake x

@property
property2 : Bool
property2 = forall x .
  validInput x and
  not (safeBack x) =>
  actionToTake accelerate x



---
## Example

In [45]:
! vehicle verify \
    --specification car-safety.vcl \
    --verifier Marabou \
    --network nnModel:nn_model_finetuned.onnx \
    --property property1 \
    # --property property2



In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the verifier, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.

Verifying properties:
  property1 [......................................................] 0/4 queries
    result: ✗ - Marabou found a counterexample
      x: [ 3.99997824384, 46.48166112692, 0.539032764554, 2.633688732782 ]


For this counterexample, the model should have predicted "brake":
$$ d_{front} = 4.0 $$
$$ L + v_{max}^2/2B_{max} = 4.0 + 400/10 = 44.0 > d_{front} $$
$$ => not (safeFront) $$

Instead, the model picked...

In [46]:
# From our .vcl file
meanScalingValues = np.array([68.479297, 69.437695, 0.393321, 0.718784], dtype=np.float32)
standardDeviation = np.array([52.128595, 52.274740, 2.142253, 1.982537], dtype=np.float32)

actions = ["brake", "idle", "accelerate"]
brake, idle, accelerate = 0, 1, 2

# Counterexample from prpoerty1 above
x_counterex = np.array([
    3.99997824384,     # d_front
    46.48166112692,    # d_back
    0.539032764554,    # v_front
    2.63368873278      # v_back
], dtype=np.float32)

# Property 2 counterexample
# x_counterex = np.array([
#     21.003648261105,     # d_front
#     4.00001864378,    # d_back
#     1.092858721379,    # v_front
#     2.420702800335      # v_back
# ], dtype=np.float32)

x_norm = (x_counterex - meanScalingValues) / standardDeviation
x_norm = x_norm.reshape(1, -1)

session = ort.InferenceSession("nn_model_finetuned.onnx")
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

# Run inference
output = session.run([output_name], {input_name: x_norm})[0][0]

predicted_action_idx = int(np.argmax(output))
predicted_action = actions[predicted_action_idx]

print("Network output logits:")
for i, a in enumerate(actions):
    print(f"  {a:10s}: {output[i]: .5f}")

print(f"\nPredicted action: {predicted_action.upper()}")


Network output logits:
  brake     :  0.71914
  idle      :  1.38056
  accelerate: -6.53132

Predicted action: IDLE


In [47]:
! vehicle compile \
    --target MarabouQueries \
    --specification car-safety.vcl \
    --network nnModel:nn_model_finetuned.onnx \
    --parameter epsilon:0.05 \
    --output query_cache




In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the verifier, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.



In order to provide support, Vehicle has automatically converted the strict inequalities to non-strict inequalites. This is not sound, but errors will be at most the floating point epsilon used by the verifier, which is usually very small (e.g. 1e-9). However, this may lead to unexpected behaviour (e.g. loss of the law of excluded middle).

See https://github.com/vehicle-lang/vehicle/issues/74 for further details.

